<a href="https://colab.research.google.com/github/usharani-192312304/ITA0605-ML/blob/main/Exp07_Logistic_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 7: Logistic Regression (Social Network Ads)
**Course:** ITA06 - Machine Learning (Lab)

## Aim
Write a program to implement the Logistic Regression (LR) algorithm in Python.

## Background
Logistic regression models the probability of a binary outcome with the **sigmoid** function: `P(y=1|x) = 1 / (1 + e^-(w.x + b))`. The weights are learned by minimising the **log-loss** (binary cross-entropy) with gradient descent. A threshold (default 0.5) turns probabilities into class labels. The decision boundary `w.x + b = 0` is linear.

## Pipeline followed in this notebook
1. Step 1: Data Collection
2. Step 2: Data Loading
3. Step 3: Data Preprocessing
4. Step 4: Exploratory Data Analysis (EDA)
5. Step 5: Model Training
6. Step 6: Model Evaluation
7. Step 7: Prediction on New Data

> **How to run:** Open in Google Colab -> *Runtime -> Run all*. No files need to be uploaded; the data is created or loaded inside the notebook.

## Setup: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
np.random.seed(42)
print("Libraries imported successfully")

## Step 1: Data Collection
We generate a *Social Network Ads*-style dataset: 400 users with **Age**, **Estimated Salary** and **Gender**; the target is whether the user **purchased** the advertised product. It is saved as `social_network_ads.csv`.

In [ ]:
rng = np.random.RandomState(42)
n = 400
zs = lambda v: (v - v.mean()) / v.std()

age = rng.randint(18, 60, n)
salary = np.clip(rng.normal(70000, 34000, n), 15000, 150000).round(-3)
gender = rng.choice(["Male", "Female"], n)

score = 0.9 * zs(age) + 0.7 * zs(salary)
prob = 1 / (1 + np.exp(-(-0.7 + 2.2 * zs(score))))
purchased = rng.binomial(1, prob)

pd.DataFrame({"User ID": 15600000 + np.arange(n), "Gender": gender, "Age": age,
              "EstimatedSalary": salary, "Purchased": purchased}).to_csv("social_network_ads.csv", index=False)
print("Saved social_network_ads.csv")

## Step 2: Data Loading

In [ ]:
df = pd.read_csv("social_network_ads.csv")
df.head()

## Step 3: Data Preprocessing
Drop the useless ID column, encode Gender, split into train/test and scale the numeric features.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df = df.drop(columns="User ID")
df["Gender"] = df["Gender"].map({"Male": 1, "Female": 0})
print("Missing values:", df.isnull().sum().sum())

X = df[["Age", "EstimatedSalary", "Gender"]]
y = df["Purchased"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

scaler = StandardScaler().fit(X_train[["Age", "EstimatedSalary"]])
def prep(X_):
    X_ = X_.copy()
    X_[["Age", "EstimatedSalary"]] = scaler.transform(X_[["Age", "EstimatedSalary"]])
    return X_
X_train_s, X_test_s = prep(X_train), prep(X_test)
print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

## Step 4: Exploratory Data Analysis (EDA)

In [ ]:
print(df.describe().T)
print("\nPurchase rate:", df["Purchased"].mean().round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.countplot(data=df, x="Purchased", ax=axes[0]); axes[0].set_title("Class balance")
sns.histplot(data=df, x="Age", hue="Purchased", kde=True, ax=axes[1]); axes[1].set_title("Age vs Purchase")
sns.scatterplot(data=df, x="Age", y="EstimatedSalary", hue="Purchased", ax=axes[2]); axes[2].set_title("Age vs Salary")
plt.tight_layout(); plt.show()
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0); plt.title("Correlation"); plt.show()

## Step 5: Model Training
Logistic regression **from scratch** with batch gradient descent, then scikit-learn's `LogisticRegression`.

In [ ]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.1, epochs=2000):
        self.lr, self.epochs = lr, epochs

    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        X, y = np.asarray(X, float), np.asarray(y, float)
        self.w, self.b = np.zeros(X.shape[1]), 0.0
        self.loss_history = []
        for _ in range(self.epochs):
            p = self.sigmoid(X @ self.w + self.b)
            self.w -= self.lr * X.T @ (p - y) / len(y)        # gradient of log-loss
            self.b -= self.lr * np.mean(p - y)
            p = np.clip(p, 1e-12, 1 - 1e-12)
            self.loss_history.append(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))
        return self

    def predict_proba(self, X):
        return self.sigmoid(np.asarray(X, float) @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

scratch = LogisticRegressionScratch().fit(X_train_s, y_train)
plt.plot(scratch.loss_history); plt.xlabel("Iteration"); plt.ylabel("Log-loss"); plt.title("Gradient descent convergence"); plt.show()

from sklearn.linear_model import LogisticRegression
model = LogisticRegression().fit(X_train_s, y_train)
print("sklearn coefficients:", dict(zip(X.columns, model.coef_[0].round(3))), "| intercept:", model.intercept_.round(3))
print("scratch coefficients:", dict(zip(X.columns, scratch.w.round(3))), "| intercept:", round(scratch.b, 3))

## Step 6: Model Evaluation
Accuracy, confusion matrix, precision/recall/F1, ROC curve, plus the **decision boundary** of a 2-feature model.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]
print("Train accuracy:", round(model.score(X_train_s, y_train), 4), "| Test accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("Scratch test accuracy:", round(accuracy_score(y_test, scratch.predict(X_test_s)), 4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))
print(classification_report(y_test, y_pred, target_names=["Not purchased", "Purchased"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual"); axes[0].set_title("Confusion matrix")
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, y_prob):.3f}"); axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR"); axes[1].set_title("ROC curve"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Decision boundary using just Age and Salary
Xtr2 = X_train_s[["Age", "EstimatedSalary"]].values
m2 = LogisticRegression().fit(Xtr2, y_train)
xx, yy = np.meshgrid(np.linspace(Xtr2[:, 0].min() - .5, Xtr2[:, 0].max() + .5, 300),
                     np.linspace(Xtr2[:, 1].min() - .5, Xtr2[:, 1].max() + .5, 300))
Z = m2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
plt.contourf(xx, yy, Z, levels=20, cmap="RdBu_r", alpha=0.6)
plt.contour(xx, yy, Z, levels=[0.5], colors="k")
plt.scatter(Xtr2[:, 0], Xtr2[:, 1], c=y_train, cmap="coolwarm", edgecolor="k", s=25)
plt.xlabel("Age (scaled)"); plt.ylabel("Salary (scaled)"); plt.title("Logistic regression decision boundary"); plt.colorbar(label="P(purchase)")
plt.show()

## Step 7: Prediction on New Data
Will a 30-year-old earning 87,000 buy? What about a 50-year-old earning 90,000?

In [ ]:
new_users = pd.DataFrame({"Age": [30, 50], "EstimatedSalary": [87000, 90000], "Gender": [1, 0]})
probs = model.predict_proba(prep(new_users))[:, 1]
for (_, row), p in zip(new_users.iterrows(), probs):
    print(f"Age {int(row.Age)}, Salary {int(row.EstimatedSalary)} -> P(purchase) = {p:.3f} -> {'Will buy' if p >= .5 else 'Will not buy'}")

## Conclusion
Logistic regression separated buyers from non-buyers with roughly 85-90% accuracy and a high ROC-AUC. Older users and higher earners are more likely to purchase (positive coefficients for Age and EstimatedSalary), while Gender has almost no effect. The scratch gradient-descent version converged to nearly the same weights as scikit-learn.